## Function: Setup OSM and PostGIS 🗄️

In this notebook, you'll learn how to build the `setup_osm_postgis()` function step by step. This workflow sets up a spatial database, downloads Geofabrik shapefile data, and loads it into PostGIS.

### 🎯 What This Function Does
- Connect to a PostgreSQL server
- Create a new database
- Enable PostGIS extension
- Download shapefile data from Geofabrik
- Unzip shapefile data
- Load shapefiles into PostGIS

### 🔧 Function Signature
```python
def setup_osm_postgis(osm_url: str,
    db_name: str = "osm_db",
    user: str = "postgres",
    password: str = "postgres",
    host: str = "localhost",
    port: int = 5432,
    data_dir: Optional[Path] = None):
    """      
    Args:
    osm_url: URL to Geofabrik shapefile ZIP
    db_name: Name of the database to create
    user: PostgreSQL username
    password: PostgreSQL password
    host: Database host
    port: Database port
    data_dir: Directory to store downloaded OSM data

    Returns:
        None
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/setup_osm_postgis.py`  
**Function Name**: `setup_osm_postgis()`  
**Replace**: The placeholder function with your working code

---

### ⚙️ Step 0: Select the Correct Python Kernel

Before running any cells, make sure the notebook is using the correct Python environment.

**Check the kernel in the top-right corner of the notebook.**

The correct Python environment is **python-gis-postgis-development (.venv)**  
It may appear with a Python version, for example:  
**python-gis-postgis-development (Python 3.11.15)**

If the kernel is **python-gis-postgis-development (Python 3.11.15)**, you can start running cells below.

Steps to select the correct kernel:
1. Click on the kernel (top right corner of the notebook) if it is not **python-gis-postgis-development (Python 3.11.15)** or if it says "Select Kernel"
2. Select **python-gis-postgis-development (Python 3.11.15)**
3. If you do not see the kernel in the list, click on "Select Another Kernel..."  
    a. Click on Python Environments...  
    b. Select **python-gis-postgis-development (Python 3.11.15)**

Once the correct kernel is selected, you can start running cells below.

### 📚 Step 1: Import Required Libraries

We will use the following tools:
- `psycopg2`: connect to PostgreSQL
- `requests`: download OSM data
- `subprocess`: run system commands (`shp2pgsql`)
- `os`: manage file paths
- `zipfile`: unzip Zip files

**💡 These will be used in our function!**

In [ ]:
import os
import requests
import subprocess
import psycopg2
import zipfile

print("Libraries imported!")

### 🔌 Step 2: Connect to PostgreSQL Server

 ⚠️ **Database Container Required** -
 
This notebook assumes your PostGIS database container is already running.  
You can verify it is running with:  
 
```bash
docker ps
```
If you have not started it yet, run the following in a terminal:  
  
 ```bash
docker compose up -d
```  
We will first connect to the default `postgres` database.

**💡 This will be used in our function!**

In [ ]:
# Establish a connection to the postgres database
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

# Every SQL statement is executed immediately
conn.autocommit = True

# Open a cursor to perform database operations
cur = conn.cursor()

# Check current database by using PostgreSQL built-in function
cur.execute("SELECT current_database();")

# Use fetchone() to return one row from the result of the query
print("Current database:", cur.fetchone()[0])

print("Connected to PostgreSQL server")

### 🗄️ Step 3: Create a New Database

We will create a new database `arizona` where OSM data will be stored.

**💡 This will be used in our function!**

In [ ]:
# Variable stores the new database name
db_name = "arizona"

# Check if database name already exists
cur.execute(f"SELECT 1 FROM pg_database WHERE datname = '{db_name}';")

# fetchone() returns a row if it exists, otherwise None
exists = cur.fetchone()

if not exists:
    cur.execute(f"CREATE DATABASE {db_name};")
    print(f"Database '{db_name}' created")
else:
    print(f"Database '{db_name}' already exists")

# Close connection to 'postgres' before switching databases
cur.close()
conn.close()

print("Closed connection to 'postgres'")

### 🌍 Step 4: Enable PostGIS Extension

PostGIS adds spatial capabilities to PostgreSQL. Without this step, we cannot store or analyze spatial data.

**💡 This will be used in our function!**

In [ ]:
# Establish a connection to the working database
conn = psycopg2.connect(
    dbname=db_name,
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

# Every SQL statement is executed immediately
conn.autocommit = True

# Open a cursor to perform database operations
cur = conn.cursor()

print(f"Connected to database: {db_name}")

# Create extension if it does not exist
cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")

# Check PostGIS version
cur.execute("SELECT PostGIS_version();")
version = cur.fetchone()
print("PostGIS version:", version)

### ⬇️ Step 5: Download OSM Shapefile Data from Geofabrik

Geofabrik provides pre-packaged OpenStreetMap data as **shapefiles**, which are easier to use for GIS workflows. We will download Arizona data as our example.

These shapefiles are derived from OpenStreetMap and organized into layers such as roads, places, and waterways.

**💡 This will be used in our function!**

In [ ]:
# URL for OSM data download (Geofabrik - shapefile)
osm_url = "https://download.geofabrik.de/north-america/us/arizona-latest-free.shp.zip"

# Define local directory to store OSM data
data_path = f"../data/{db_name}"

# Create directory if it does not exist
os.makedirs(data_path, exist_ok=True)

# Construct full file path using the filename from the URL
zip_path = os.path.join(data_path, osm_url.split("/")[-1])

# Download file only if it does not already exist
if not os.path.exists(zip_path):
    print("Downloading OSM data...")
    print("URL:", osm_url)

    # Send HTTP request (stream=True downloads in chunks)
    response = requests.get(osm_url, stream=True, timeout=300)
    # Raise error if download fails
    response.raise_for_status()
    
    # Get total file size (if available) for progress tracking
    file_size = int(response.headers.get("content-length", 0))
    if file_size > 0:
        print(f"File size: {file_size / (1024 * 1024):.1f} MB")

    downloaded = 0
    
    # Open file in binary write mode
    with open(zip_path, "wb") as f:
        # Download file in chunks to avoid loading entire file into memory
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                
                # Display progress as percentage (if file size is known)
                if file_size > 0:
                    progress = downloaded / file_size * 100
                    print(f"\rProgress: {progress:.1f}%", end="", flush=True)

    print("\nDownload complete")
    print("Saved to:", zip_path)
else:
    # Skip download if file already exists locally
    print("File already exists:")
    print(zip_path)

### 📂 Step 6: Unzip the Downloaded OSM Zip File

We will unzip the shapefile from the zip file.

**💡 This will be used in our function!**

In [ ]:
# Extract ZIP contents
extract_path = os.path.join(data_path, "shapefiles")

if not os.path.exists(extract_path):
    print("Extracting shapefiles...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete")
    print("Extracted to:", extract_path)
else:
    print("Extracted folder already exists:")
    print(extract_path)

### 📥 Step 7: Load Shapefile Data into PostGIS

We use `shp2pgsql` to load shapefile data into PostGIS database tables. This command runs in a Unix shell and converts shapefiles into SQL statements that are executed in the database.  
Only selected shapefile layers are loaded to improve performance and focus on key analysis datasets

**Command Explanation:**
- `shp2pgsql`: converts shapefiles into SQL
- `-d`: drops the table if it already exists
- `-I`: creates a spatial index
- `-s 4326`: sets the coordinate reference system (WGS84)
- `public.table_name`: destination table in PostGIS
- `psql`: executes the SQL commands in the database

**💡 This will be used in our function!**

In [ ]:
# List of shapefiles to load
load_shapefiles = [
    "places_a", 
    "railways",
    "landuse_a",
    "pois",
    "adminareas_a",
    "roads"]

# Set password so shp2pgsql/psql does not prompt
env = os.environ.copy()
env["PGPASSWORD"] = "postgres"

# Find all .shp files
shp_files = [f for f in os.listdir(extract_path) if f.endswith(".shp")]

# Loop through shapefiles and load them
for shp_file in shp_files:
    filename = os.path.splitext(shp_file)[0]

    table_name = (
        filename
        .replace("gis_osm_", "")
        .replace("_free_1", "")
    )

    # Skip shapefiles that are not in our load list
    if table_name not in load_shapefiles: continue

    shp_path = os.path.join(extract_path, shp_file)

    print(f"\nLoading {table_name} from {filename}...")

    cmd = f'shp2pgsql -d -I -s 4326 "{shp_path}" public.{table_name} | psql -h localhost -U postgres -d {db_name}'

    print("Command:", cmd)

    try:
        subprocess.run(cmd, shell=True, env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        print(f"{table_name} loaded successfully")
    except subprocess.CalledProcessError as e:
        print(f"{table_name} failed")
        print(e.stderr.splitlines()[-1])  # show only last error line

### ✅ Step 8: Verify Shapefile Data Load

We now confirm that the Geofabrik shapefile data was successfully loaded into PostGIS.

**What this check does:**
- lists all tables created in the database  
- verifies that shapefile layers were loaded into PostGIS  
- counts the number of rows in each table  

**💡 If expected tables are missing or row counts are zero, the data load did not complete correctly**

In [ ]:
# Create an SQL query to list all tables
list_query = """
SELECT table_name 
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

# Execute the query
cur.execute(list_query)

# Retrieve all table names
tables = [t[0] for t in cur.fetchall()]
print("Tables created:", tables)

# Row counts for all tables (not hardcoded)
for table in tables:
    try:
        cur.execute(f'SELECT COUNT(*) FROM "{table}";')
        count = cur.fetchone()[0]
        print(f"{table}: {count} rows")
    except Exception as e:
        print(f"{table}: error")
        print(e)

# Close the database connection
cur.close()
conn.close()

print("Database connection closed")

### 🏗️ Step 9: Build the Complete Function

Now let's put everything together into a reusable function. This is what you will implement in `src/setup_osm_postgis.py`:

Find this:
```python
def setup_osm_postgis(...):
    # TODO: Implement this function
    # Step 1: Setup data directory
    # Step 2: Download shapefile ZIP data
    # ... (remaining TODO steps)
```
**⚠️ Before implementing your code, remove the raise NotImplementedError(...) line.**

Then replace ALL the TODO comments with your working code from the steps above.

In [ ]:
def setup_osm_postgis(
    osm_url: str,
    db_name: str = "osm_db",
    user: str = "postgres",
    password: str = "postgres",
    host: str = "localhost",
    port: int = 5432,
    data_dir: Optional[Path] = None,
    load_shapefiles: Optional[list[str]] = None
) -> None:

    """
    Set up a PostGIS database and load OSM shapefile data from Geofabrik.

    Args:
        osm_url: URL to Geofabrik shapefile ZIP
        db_name: Name of the database to create/use
        user: PostgreSQL username
        password: PostgreSQL password
        host: Database host
        port: Database port
        data_dir: Optional directory to store downloaded OSM data
        load_shapefiles: Optional list of shapefile layer names to load
    Returns:
        None
    """

    # Step 1: Setup data directory
    if data_dir is None:
        data_dir = Path(f"../data/{db_name}")
    else:
        data_dir = Path(data_dir)

    data_dir.mkdir(parents=True, exist_ok=True)

    # Construct file path from URL
    file_path = data_dir / osm_url.split("/")[-1]

    # Step 2: Download shapefile ZIP data
    if not file_path.exists():
        print("Downloading shapefile data...")
        print("URL:", osm_url)

        response = requests.get(osm_url, stream=True, timeout=300)
        response.raise_for_status()

        # Get file size for progress display
        file_size = int(response.headers.get("content-length", 0))
        if file_size > 0:
            print(f"File size: {file_size / (1024 * 1024):.1f} MB")

        downloaded = 0

        # Download in chunks
        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

                    # Show progress percentage
                    if file_size > 0:
                        progress = downloaded / file_size * 100
                        print(f"\rProgress: {progress:.1f}%", end="", flush=True)

        print("\nDownload complete")
        print("Saved to:", file_path)
    else:
        print("File already exists:")
        print(file_path)

    # Step 3: Connect to PostgreSQL (default database)
    conn = psycopg2.connect(
        dbname="postgres",
        user=user,
        password=password,
        host=host,
        port=port
    )

    # Execute SQL immediately
    conn.autocommit = True

    cur = conn.cursor()

    print("Connected to PostgreSQL server")

    # Step 4: Create the working database
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s;", (db_name,))
    exists = cur.fetchone()

    if not exists:
        cur.execute(f'CREATE DATABASE "{db_name}";')
        print(f"Database '{db_name}' created")
    else:
        print(f"Database '{db_name}' already exists")

    # Verify database exists
    cur.execute("SELECT datname FROM pg_database WHERE datname = %s;", (db_name,))
    result = cur.fetchone()
    if result:
        print("Verified:", result[0])

    # Close connection before switching databases
    cur.close()
    conn.close()
    print("Closed connection to 'postgres'")

    # Step 5: Connect to the new database
    conn = psycopg2.connect(
        dbname=db_name,
        user=user,
        password=password,
        host=host,
        port=port
    )

    conn.autocommit = True
    cur = conn.cursor()

    print("Connected to database:", db_name)

    # Step 6: Enable PostGIS
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")

    # Check PostGIS installation
    cur.execute("SELECT PostGIS_version();")
    version = cur.fetchone()[0]
    print("PostGIS version:", version)

    # Step 7: Unzip shapefile data
    extract_path = data_dir / "shapefiles"

    if not extract_path.exists():
        print("Extracting shapefiles...")
        with zipfile.ZipFile(file_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)
        print("Extraction complete")
        print("Extracted to:", extract_path)
    else:
        print("Extracted folder already exists:", extract_path)

    # Step 8: Load shapefiles into PostGIS using shp2pgsql
      
    env = os.environ.copy()
    env["PGPASSWORD"] = password

    shp_files = [f for f in os.listdir(extract_path) if f.endswith(".shp")]
    
    for shp_file in shp_files:
        filename = os.path.splitext(shp_file)[0]

        table_name = (
            filename
            .replace("gis_osm_", "")
            .replace("_free_1", "")
        )

        # Skip if filtering is enabled
        if load_shapefiles is not None and table_name not in load_shapefiles:
            continue

        shp_path = os.path.join(extract_path, shp_file)

        print(f"\nLoading {table_name} from {filename}...")

        cmd = f'shp2pgsql -d -I -s 4326 "{shp_path}" public.{table_name} | psql -h {host} -U {user} -d {db_name}'

        print("Command:", cmd)

        try:
            subprocess.run(cmd, shell=True, env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
            print(f"{table_name} loaded successfully")
        except subprocess.CalledProcessError as e:
            print(f"{table_name} failed")
            print(e.stderr.splitlines()[-1])

    # Step 9: Close connections
    cur.close()
    conn.close()

    print("Database connection closed")

### 🔑 Key Learning Points

- `psycopg2.connect()` is used to connect Python to a PostgreSQL database  
- `cursor()` allows you to execute SQL queries from Python  
- `fetchone()` and `fetchall()` retrieve query results from the database  
- PostgreSQL system tables (e.g., `pg_database`) can be used to inspect database state  
- `CREATE DATABASE` and `CREATE EXTENSION` require autocommit mode  
- PostGIS enables spatial data types and spatial queries in PostgreSQL  
- Geofabrik provides OpenStreetMap data as preprocessed shapefiles for GIS workflows  
- Shapefiles are commonly distributed as `.zip` archives and must be extracted before use  
- `requests` can be used to download large files over HTTP  
- `shp2pgsql` converts shapefiles into SQL and loads them into PostGIS tables  
- `subprocess.run()` allows Python to execute shell commands  
- Always verify your workflow (tables + row counts) to confirm data loaded correctly  
- Close database connections (`cur.close()`, `conn.close()`) to release resources  